### AST Parsing

In [7]:
import json
import os
from ast_parser import parse_project

# Define the project directory manually in the notebook
project_root = "P:\My Project\PdfViewer"   # <-- change this

# Validate directory
if not os.path.isdir(project_root):
    raise ValueError(f"Error: '{project_root}' is not a valid directory")

# Parse AST
ast_data = parse_project(project_root)

# Store output in the CURRENT working directory
project_name = os.path.basename(os.path.abspath(project_root))
output_file = os.path.join(os.getcwd(), f"{project_name}_ast.json")

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(ast_data, f, indent=2)

print("AST parsing completed successfully ✅")
print(f"AST data saved to: {output_file}")

<>:6: SyntaxWarning: invalid escape sequence '\M'
<>:6: SyntaxWarning: invalid escape sequence '\M'
C:\Users\anura\AppData\Local\Temp\ipykernel_41916\1687151003.py:6: SyntaxWarning: invalid escape sequence '\M'
  project_root = "P:\My Project\PdfViewer"   # <-- change this


AST parsing completed successfully ✅
AST data saved to: p:\8th Sem\Neural Networks\Project\PdfViewer_ast.json


### Code Graph Building

In [8]:
import json
import os
from code_graph_builder import CodeGraphBuilder

# Define the AST JSON file path manually in the notebook
ast_file = r"P:\8th Sem\Neural Networks\Project\PdfViewer_ast.json"   # <-- change this

# Validate file
if not os.path.isfile(ast_file):
    raise FileNotFoundError(f"Error: '{ast_file}' does not exist")

# Load AST data
with open(ast_file, "r", encoding="utf-8") as f:
    ast_data = json.load(f)

# Build the code property graph
builder = CodeGraphBuilder()
graph = builder.build(ast_data)

print("Code Property Graph created successfully ✅")
print("Nodes:", graph.number_of_nodes())
print("Edges:", graph.number_of_edges())


Building code property graph...
[+] Added file nodes
[+] Added class and function nodes
[+] Built lookup indices
[+] Added DEFINES edges
[+] Added IMPORT edges
[+] Added INHERITANCE edges
[+] Added CALL edges
[!] Large graph detected (21232 nodes). Using optimized cycle detection...
[!] For large graphs, applying heuristic-based cycle breaking...
[OK] Heuristic cycle breaking completed
[WARNING] Graph still contains cycles after attempt to break them
Code Property Graph created successfully ✅
Nodes: 21232
Edges: 57171


### Node Feature Engineering

In [9]:
import importlib
import sys
import time

# Reload modules to get the latest version
if 'node_feature_engineering' in sys.modules:
    del sys.modules['node_feature_engineering']

from node_feature_engineering import extract_node_features

print("Starting feature extraction...")
start = time.time()

node_features = extract_node_features(graph)

elapsed = time.time() - start
print(f"\nTotal nodes: {len(node_features)}")
print(f"Feature length: {len(next(iter(node_features.values())))}")
print(f"Time taken: {elapsed:.2f} seconds")
print(f"\nFeature vector for first node:")
first_node = list(node_features.keys())[0]
print(f"Node: {first_node}")
print(f"Features: {node_features[first_node]}")

Starting feature extraction...
Computing features for 21232 nodes...
✅ Depths computed for all nodes
   Processing node 0/21232...
   Processing node 5000/21232...
   Processing node 10000/21232...
   Processing node 15000/21232...
   Processing node 20000/21232...
✅ Features extracted successfully

Total nodes: 21232
Feature length: 8
Time taken: 0.08 seconds

Feature vector for first node:
Node: FILE::app.py
Features: [1, 0, 0, 0, 0, 0, 114, 0]


### Graph Neural Network

In [10]:
import importlib
import sys
import time

# Reload module to get latest version
if 'gnn_model' in sys.modules:
    del sys.modules['gnn_model']

from gnn_model import prepare_pyg_data, train_gnn, get_embeddings

# ---- Prepare PyG Data ----
print("="*60)
print("Section 6.4: Graph Neural Network (GraphSAGE)")
print("="*60)

data, node_mapping = prepare_pyg_data(graph, node_features)

# ---- Train the GNN ----
start = time.time()
model, losses = train_gnn(data, epochs=100, lr=0.01)
train_time = time.time() - start

# ---- Extract Node Embeddings ----
embeddings = get_embeddings(model, data, node_mapping)

# ---- Summary ----
print(f"\n{'='*60}")
print(f"Results Summary")
print(f"{'='*60}")
print(f"  Nodes embedded:      {len(embeddings):,}")
print(f"  Embedding dimension: {len(next(iter(embeddings.values())))}")
print(f"  Training time:       {train_time:.2f} seconds")
print(f"  Final loss:          {losses[-1]:.4f}")

# Show sample embeddings
print(f"\nSample embeddings (first 3 nodes):")
for i, (node, emb) in enumerate(embeddings.items()):
    if i >= 3:
        break
    emb_str = ', '.join(f'{v:.4f}' for v in emb[:5])
    print(f"  {node}")
    print(f"    [{emb_str}, ...]")

print(f"\n✅ Section 6.4 complete!")

Section 6.4: Graph Neural Network (GraphSAGE)
Preparing PyG data...
  Nodes with features: 21232
  Edges (undirected): 114294
  Feature dimension: 8
✅ PyG Data object created

Training GraphSAGE model...
  Architecture: 8 → 64 → 32
  Positive edges: 114294
  Epochs: 100, LR: 0.01

  Epoch   1/100 | Loss: 7832.2637
  Epoch  10/100 | Loss: 914.8668
  Epoch  20/100 | Loss: 463.6338
  Epoch  30/100 | Loss: 229.1263
  Epoch  40/100 | Loss: 182.1635
  Epoch  50/100 | Loss: 132.9705
  Epoch  60/100 | Loss: 84.1913
  Epoch  70/100 | Loss: 72.0168
  Epoch  80/100 | Loss: 58.3017
  Epoch  90/100 | Loss: 47.8269
  Epoch 100/100 | Loss: 39.0381

✅ Training complete! Final loss: 39.0381
✅ Extracted embeddings for 21232 nodes
   Embedding dimension: 32

Results Summary
  Nodes embedded:      21,232
  Embedding dimension: 32
  Training time:       10.60 seconds
  Final loss:          39.0381

Sample embeddings (first 3 nodes):
  FILE::app.py
    [0.2548, -0.1496, -0.2814, 0.2032, -0.0444, ...]
  FILE

### Component Inference — COPY THIS INTO NOTEBOOK

In [11]:
import importlib
import sys

# Reload module to get latest version
if 'component_inference' in sys.modules:
    del sys.modules['component_inference']

from component_inference import infer_components_kmeans, evaluate_clusters, assign_components_to_graph

print("="*60)
print("Section 6.5: Component Inference")
print("="*60)

# Run K-Means Clustering on the embeddings
# (Assuming 'embeddings', 'graph', and 'node_mapping' exist from Section 6.4)
n_components = 15  # Choose number of expected architectural components
labels = infer_components_kmeans(embeddings, n_clusters=n_components)

# Evaluate the clusters
score = evaluate_clusters(embeddings, labels)

# Assign the component IDs back to the original graph attributes
assign_components_to_graph(graph, labels, node_mapping)

# Print distribution of nodes across components
from collections import Counter
counts = Counter(labels.values())

print(f"\nComponent Distribution (Top 10):")
for comp_id, count in counts.most_common(10):
    print(f"  Component {comp_id:2d}: {count:5d} nodes")

print(f"\n✅ Section 6.5 complete!")

Section 6.5: Component Inference
Running K-Means clustering (K=15)...
✅ Grouped 21232 nodes into 15 components
✅ Silhouette Score: 0.2704 (range: -1 to 1, >0.5 is good)
✅ Assigned component IDs to 21232/21232 graph nodes

Component Distribution (Top 10):
  Component  9:  2173 nodes
  Component  6:  2029 nodes
  Component  2:  2015 nodes
  Component 11:  1821 nodes
  Component  5:  1779 nodes
  Component  0:  1438 nodes
  Component 10:  1363 nodes
  Component  1:  1350 nodes
  Component 12:  1338 nodes
  Component  4:  1204 nodes

✅ Section 6.5 complete!


### Component Graph Construction

In [12]:
import importlib
import sys
import os

# Reload module to get latest version
if 'component_graph' in sys.modules:
    del sys.modules['component_graph']

from component_graph import build_component_graph, visualize_architecture, visualize_tree_diagram

print("="*60)
print("Section 6.6: Component Graph Construction")
print("="*60)

# Build high-level component graph
comp_graph = build_component_graph(graph)

# Visualize the standard graph
html_path_standard = visualize_architecture(comp_graph, "architecture.html")

# Visualize the hierarchical tree/block diagram
html_path_tree = visualize_tree_diagram(comp_graph, "architecture_tree.html")

# Provide IPython display links
from IPython.display import IFrame, display, HTML

display(HTML(f"<h3>Standard Architecture Diagram</h3>"))
display(HTML(f"<a href='architecture.html' target='_blank'>Open in new tab</a>"))
display(IFrame(src='./architecture.html', width='100%', height='500px'))

display(HTML(f"<h3>Hierarchical Tree (Block) Diagram</h3>"))
display(HTML(f"<a href='architecture_tree.html' target='_blank'>Open in new tab</a>"))
display(IFrame(src='./architecture_tree.html', width='100%', height='500px'))

print(f"\n✅ Section 6.6 complete!")



Section 6.6: Component Graph Construction
Building high-level Component Graph...
✅ Collapsed 21232 low-level nodes into 15 architectural components.
✅ Found 184 interactions between components.
Visualizing architecture components...
✅ Interactive architecture diagram saved to: p:\8th Sem\Neural Networks\Project\architecture.html
Visualizing hierarchical tree diagram...
✅ Hierarchical tree diagram saved to: p:\8th Sem\Neural Networks\Project\architecture_tree.html



✅ Section 6.6 complete!
